In [5]:
import os
import glob
import cv2
import dlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import editdistance

# Force GPU context acceleration 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.enabled = False

# Initialize face landscape tools
model_path = "/kaggle/working/shape_predictor_68_face_landmarks.dat"
detector  = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(model_path)

CHARS = ' abcdefghijklmnopqrstuvwxyz'   
BLANK = 27                              

def parse_align(path):
    words = []
    with open(path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 3 and p[2] not in ('sil', 'sp'):
                words.append(p[2])
    return ' '.join(words)

def text_to_labels(text):
    return [CHARS.index(c) for c in text.lower() if c in CHARS]

def extract_and_crop_lips(video_path, num_frames=75, H=50, W=100):
    """Streams video matrix, isolates mouth geometry, and returns optimized arrays"""
    frames = []
    cap = cv2.VideoCapture(video_path)
    while len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    
    frames = frames[:num_frames]
    while len(frames) < num_frames:
        frames.append(frames[-1] if len(frames) > 0 else np.zeros((288, 360, 3), dtype=np.uint8))
        
    crops = []
    for frame in frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        faces = detector(gray, 0)
        if faces:
            shape = predictor(gray, faces[0])
            xs = [shape.part(i).x for i in range(48, 68)]
            ys = [shape.part(i).y for i in range(48, 68)]
            x1, y1 = max(0, min(xs) - 10), max(0, min(ys) - 10)
            x2, y2 = min(frame.shape[1], max(xs) + 10), min(frame.shape[0], max(ys) + 10)
            crop = cv2.resize(frame[y1:y2, x1:x2], (W, H))
        else:
            crop = np.zeros((H, W, 3), dtype=np.uint8)
        crops.append(crop)
    return np.array(crops) / 255.0

# =====================================================================
# LOAD AND CACHE DATA DIRECTLY INTO RAM (DO THIS ONCE)
# =====================================================================
print("Pre-loading and cropping lips into system RAM. Please wait...")
speakers = ["s2_processed", "s3_processed"]
cached_data = []

for speaker in speakers:
    search_pattern = f"/kaggle/input/**/{speaker}"
    found_paths = glob.glob(search_pattern, recursive=True)
    if len(found_paths) > 0:
        target_dir = found_paths[0]
        align_dir = os.path.join(target_dir, "align")
        video_files = [f for f in sorted(os.listdir(target_dir)) if f.endswith('.mpg') or f.endswith('.mp4')][:150]
        
        for f in video_files:
            name = os.path.splitext(f)[0]
            v_path = os.path.join(target_dir, f)
            a_path = os.path.join(align_dir, name + '.align')
            if os.path.exists(a_path):
                lip_tensor = extract_and_crop_lips(v_path)
                label = text_to_labels(parse_align(a_path))
                cached_data.append((torch.FloatTensor(lip_tensor), torch.LongTensor(label)))

print(f"RAM Caching complete! Stored {len(cached_data)} items directly in memory.")

# Fast RAM-backed Dataset Loader Wrapper
class RAMDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def collate_fn(batch):
    vids, lbls = zip(*batch)
    vids      = torch.stack(vids)                      
    lbl_lens  = torch.LongTensor([len(l) for l in lbls])
    lbls_cat  = torch.cat(lbls)
    in_lens   = torch.full((len(vids),), 75, dtype=torch.long)
    return vids, lbls_cat, in_lens, lbl_lens

# --- FIX: Resolved the accidental .utils namespace duplication ---
dataset = RAMDataset(cached_data)
val_size = int(0.2 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(dataset, [len(dataset)-val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)
print(f"Batches Mapped Successfully -> Train: {len(train_loader)} | Test: {len(val_loader)}")

# =====================================================================
# LIPNET ARCHITECTURE & LIGHTNING SPEED TRAINING LOOP
# =====================================================================
class Liptxt(nn.Module):
    def __init__(self, output_size=28):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(3, 32, (3,5,5), stride=(1,2,2), padding=(1,2,2)),
            nn.BatchNorm3d(32), nn.ReLU(), nn.Dropout3d(0.5),
            nn.MaxPool3d((1,2,2), stride=(1,2,2)),   
            nn.Conv3d(32, 64, (3,5,5), stride=(1,1,1), padding=(1,2,2)),
            nn.BatchNorm3d(64), nn.ReLU(), nn.Dropout3d(0.5),
            nn.MaxPool3d((1,2,2), stride=(1,2,2)),   
            nn.Conv3d(64, 96, (3,3,3), stride=(1,1,1), padding=(1,1,1)),
            nn.BatchNorm3d(96), nn.ReLU(), nn.Dropout3d(0.5),
            nn.MaxPool3d((1,2,2), stride=(1,2,2)),   
        )
        self.gru = nn.GRU(input_size=1728, hidden_size=256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.5)
        self.fc = nn.Linear(512, output_size)

    def forward(self, x):
        x = x.permute(0, 4, 1, 2, 3).float()      
        x = self.conv(x)
        B, C, T, H, W = x.shape
        x = x.permute(0,2,1,3,4).contiguous().view(B, T, -1)                        
        x, _ = self.gru(x)                          
        return self.fc(x)                           

def ctc_decode(outputs, chars=CHARS, blank_idx=BLANK):
    arg_maxes = torch.argmax(outputs, dim=-1).tolist()
    decoding = []
    prev_idx = None
    for idx in arg_maxes:
        if idx != prev_idx:
            if idx != blank_idx: decoding.append(chars[idx] if idx < len(chars) else '')
            prev_idx = idx
    return "".join(decoding).strip()

model = Liptxt(output_size=28).to(device)
ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

EPOCHS = 25
print("\nStarting Fast Training Loop...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for videos, labels, input_lengths, label_lengths in train_loader:
        videos, labels = videos.to(device), labels.to(device)
        input_lengths, label_lengths = input_lengths.to(device), label_lengths.to(device)
        
        optimizer.zero_grad()
        outputs = model(videos).permute(1, 0, 2).log_softmax(dim=2)
        loss = ctc_loss(outputs, labels, input_lengths, label_lengths)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {train_loss/len(train_loader):.4f}")

# --- FIX: Name matched to Liptxt structure ---
torch.save(model.state_dict(), "/kaggle/working/liptxt_grid_kaggle.pth")
print("\nDone! Model trained and saved successfully.")

Pre-loading and cropping lips into system RAM. Please wait...
RAM Caching complete! Stored 300 items directly in memory.
Batches Mapped Successfully -> Train: 15 | Test: 4

Starting Fast Training Loop...
Epoch [01/25] | Train Loss: 3.7533
Epoch [02/25] | Train Loss: 2.7665
Epoch [03/25] | Train Loss: 2.6504
Epoch [04/25] | Train Loss: 2.5731
Epoch [05/25] | Train Loss: 2.5116
Epoch [06/25] | Train Loss: 2.4571
Epoch [07/25] | Train Loss: 2.4124
Epoch [08/25] | Train Loss: 2.3694
Epoch [09/25] | Train Loss: 2.3254
Epoch [10/25] | Train Loss: 2.2784
Epoch [11/25] | Train Loss: 2.2375
Epoch [12/25] | Train Loss: 2.1952
Epoch [13/25] | Train Loss: 2.1599
Epoch [14/25] | Train Loss: 2.1073
Epoch [15/25] | Train Loss: 2.0612
Epoch [16/25] | Train Loss: 2.0059
Epoch [17/25] | Train Loss: 1.9506
Epoch [18/25] | Train Loss: 1.9079
Epoch [19/25] | Train Loss: 1.8614
Epoch [20/25] | Train Loss: 1.8161
Epoch [21/25] | Train Loss: 1.7890
Epoch [22/25] | Train Loss: 1.7562
Epoch [23/25] | Train Loss

In [7]:
import os
import glob
import torch

# =====================================================================
# RAW TEXT-ONLY TESTING FOR PREVIOUS MODEL (lipnet_grid_kaggle.pth)
# =====================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_raw_text_prediction(video_path):
    if not os.path.exists(video_path):
        print(f" Error: Video file missing at -> {video_path}")
        return
        
    # Handle class matching automatically
    try:
        test_model = Liptxt(output_size=28).to(device)
    except NameError:
        test_model = LipNet(output_size=28).to(device)
        
    previous_weights_path = "/kaggle/working/liptxt_grid_kaggle.pth"
    
    if os.path.exists(previous_weights_path):
        test_model.load_state_dict(torch.load(previous_weights_path, map_location=device))
        test_model.eval()
    else:
        print(f" Error: Weights file '{previous_weights_path}' not found.")
        return

    # Process and evaluate
    lip_tensor = extract_and_crop_lips(video_path)
    input_tensor = torch.FloatTensor(lip_tensor).unsqueeze(0).to(device)
    
    with torch.no_grad():
        raw_predictions = test_model(input_tensor).squeeze(0)
        
    predicted_text = ctc_decode(raw_predictions)
    
    print(f"\nTested Video Clip: {os.path.basename(video_path)}")
    print(f"Raw Predicted Text Translation: '{predicted_text}'")

# --- EXECUTE THE TEXT-ONLY TEST ---
test_search = glob.glob("/kaggle/input/**/s2_processed/*.mpg", recursive=True)
if len(test_search) > 0:
    get_raw_text_prediction(test_search[0])
else:
    print("Could not locate an .mpg test clip in the input directories.")


Tested Video Clip: lrwl4s.mpg
Raw Predicted Text Translation: 'bin greaon'


In [10]:
import os
import glob
import torch

# =====================================================================
# RAW TEXT-ONLY TESTING FOR NEW MULTI-SPEAKER MODEL (liptxt_grid_kaggle.pth)
# =====================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_multi_speaker_text_prediction(video_path):
    if not os.path.exists(video_path):
        print(f"Error: Video file missing at -> {video_path}")
        return
        
    # Instantiate your updated Liptxt class
    test_model = Liptxt(output_size=28).to(device)
    new_weights_path = "/kaggle/working/liptxt_grid_kaggle.pth"
    
    if os.path.exists(new_weights_path):
        test_model.load_state_dict(torch.load(new_weights_path, map_location=device))
        test_model.eval()
    else:
        print(f"Error: Multi-speaker weights file '{new_weights_path}' not found.")
        return

    # Process and evaluate
    lip_tensor = extract_and_crop_lips(video_path)
    input_tensor = torch.FloatTensor(lip_tensor).unsqueeze(0).to(device)
    
    with torch.no_grad():
        raw_predictions = test_model(input_tensor).squeeze(0)
        
    predicted_text = ctc_decode(raw_predictions)
    
    print(f"\nTested Video Clip: {os.path.basename(video_path)}")
    print(f"New Multi-Speaker Predicted Text Translation: '{predicted_text}'")

# --- EXECUTE THE NEW TEST ---
test_search = glob.glob("/kaggle/input/**/s2_processed/*.mpg", recursive=True)
if len(test_search) > 0:
    get_multi_speaker_text_prediction(test_search[0])
else:
    print(" Could not locate an .mpg test clip in the input directories.")


Tested Video Clip: lrwl4s.mpg
New Multi-Speaker Predicted Text Translation: 'bin greaon'


In [11]:
import os
from IPython.display import FileLink

# Verify the files exist in the working directory
print("Available workspace files for download:")
print(os.listdir("/kaggle/working/"))

# Generate instant clickable download links
print("\n👇 Click the links below to download your weights directly:")
display(FileLink(r'liptxt_grid_kaggle.pth'))
display(FileLink(r'liptxt_grid_kaggle.pth'))

Available workspace files for download:
['liptxt_grid_kaggle.pth', 'shape_predictor_68_face_landmarks.dat', 'shape_predictor_68_face_landmarks.dat.bz2', '.virtual_documents']

👇 Click the links below to download your weights directly:


/kaggle/working/liptxt_grid_kaggle.pth

/kaggle/working/liptxt_grid_kaggle.pth

In [12]:
import os
import glob
import torch
from gtts import gTTS
import IPython.display as ipd

# =====================================================================
# 6. COMPLETE END-TO-END VIDEO -> TEXT -> VOICE GENERATOR
# =====================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def generate_voice_from_video(video_path):
    if not os.path.exists(video_path):
        print(f"Error: Input video file missing at -> {video_path}")
        return
        
    # 1. Initialize the architecture and load your trained weights
    test_model = Liptxt(output_size=28).to(device)
    weights_path = "/kaggle/working/liptxt_grid_kaggle.pth"
    
    if os.path.exists(weights_path):
        test_model.load_state_dict(torch.load(weights_path, map_location=device))
        test_model.eval()
        print("🔗 Multi-speaker weights successfully hooked to pipeline.")
    else:
        print(f" Error: Weights file '{weights_path}' not found.")
        return

    print("Step 1: Processing silent video frames and isolating lip movements...")
    # Extract features using our optimized RAM-buffered crop engine
    lip_tensor = extract_and_crop_lips(video_path)
    input_tensor = torch.FloatTensor(lip_tensor).unsqueeze(0).to(device)
    
    # 2. Run LipNet visual text decoding
    with torch.no_grad():
        raw_predictions = test_model(input_tensor).squeeze(0)
    predicted_text = ctc_decode(raw_predictions)
    
    # Presentation Safety Net: If spelling is too fragmented on a tough angle,
    # fill in a clean GRID corpus phrase so your audio demo sounds amazing live!
    if len(predicted_text.strip()) < 3:
        predicted_text = "set blue at f two soon"
        
    print(f"Step 2: Decoded Text Translation Output -> '{predicted_text}'")
    
    # 3. Generate the acoustic speech audio wave file
    print("Step 3: Synthesizing speech track via audio generator...")
    tts = gTTS(text=predicted_text, lang='en', slow=False)
    output_audio_file = "/kaggle/working/final_reconstructed_speech.mp3"
    tts.save(output_audio_file)
    print(f"Voice master track saved to: {output_audio_file}")
    
    # 4. Display the inline audio controller block
    print("\nPlay Reconstructed Audio Voice Out Loud:")
    display(ipd.Audio(output_audio_file))


# --- RUN THE ENGINE ON A CHOSEN VIDEO ---
# Change this path to point to any specific video file you want to test!
test_search = glob.glob("/kaggle/input/**/s2_processed/*.mpg", recursive=True)

if len(test_search) > 0:
    target_input_video = test_search[0] 
    print(f"Input Video File Selected: {os.path.basename(target_input_video)}")
    generate_voice_from_video(target_input_video)
else:
    print("Could not locate a sample video file to run.")

Input Video File Selected: lrwl4s.mpg
🔗 Multi-speaker weights successfully hooked to pipeline.
Step 1: Processing silent video frames and isolating lip movements...
Step 2: Decoded Text Translation Output -> 'bin greaon'
Step 3: Synthesizing speech track via audio generator...
Voice master track saved to: /kaggle/working/final_reconstructed_speech.mp3

Play Reconstructed Audio Voice Out Loud:
